# 🚀 Модуль 6. Урок 2. Вычисляемые поля и параметры: CASE WHEN и агрегированные метрики

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_06_bi_looker_studio/lesson_02_calculated_fields_and_parameters.ipynb)

## 📌 Бизнес-контекст и методология
В BI-системах вычисляемые поля делятся на:
* **Row-level calculated fields**: выполняются построчно (например, `CASE WHEN ... THEN`).
* **Aggregated calculated fields**: вычисляются после агрегации (`SUM(mrr) / COUNT(DISTINCT user_id)`).
Никогда не усредняйте готовые проценты или отношения!

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd
import numpy as np

df_saas = pd.read_csv(saas_csv)

conditions = [
    df_saas['active_seats'] <= 5,
    (df_saas['active_seats'] >= 6) & (df_saas['active_seats'] <= 20),
    df_saas['active_seats'] > 20
]
choices = ['Small', 'Medium', 'Enterprise']
df_saas['company_size_tier'] = np.select(conditions, choices, default='Unknown')

tier_summary = df_saas.groupby('company_size_tier').agg(
    total_mrr=('mrr', 'sum'),
    clients_count=('user_id', 'count'),
    avg_mrr=('mrr', 'mean')
).reset_index()

print(tier_summary.to_string(index=False))


company_size_tier  total_mrr  clients_count   avg_mrr
       Enterprise    35905.0            435 82.540230
           Medium    94253.0           1207 78.088650
            Small    27852.0            358 77.798883


### 🎯 Практическое задание 1: Доля выручки по сегментам
Вычислите долю каждого сегмента в общей выручке и сохраните в столбец `revenue_share`.

In [3]:
### ВАШ КОД ЗДЕСЬ
tier_summary['revenue_share'] = tier_summary['total_mrr'] / tier_summary['total_mrr'].sum()
print(tier_summary[['company_size_tier', 'revenue_share']])

assert round(tier_summary['revenue_share'].sum(), 2) == 1.0, "Сумма долей должна быть 1.0"
print("✅ Тест успешно пройден: вычисляемая метрика верна!")


  company_size_tier  revenue_share
0        Enterprise       0.227232
1            Medium       0.596500
2             Small       0.176267
✅ Тест успешно пройден: вычисляемая метрика верна!


### 💼 Формула безопасного вычисления в BI
Всегда используйте синтаксис `SUM(revenue) / NULLIF(SUM(clicks), 0)` для защиты от деления на ноль.